# Logistic Regression -- UCI Bank Marketing

This notebook demonstrates logistic regression on the UCI Bank Marketing dataset, where the
goal is to predict whether a client will subscribe to a term deposit after a marketing call
based on demographic, financial, and prior-contact information.

- Train an unregularised and an L2-regularised logistic regression
- Evaluate with classification report, ROC curve, and confusion matrix
- Inspect feature coefficients to understand which variables drive subscription
- Visualise the linear decision boundary in 2-D PCA space

## Mathematical Intuition

Logistic regression models the **log-odds** of the positive class as a linear function of
the features:

$$\log \frac{P(y=1 \mid \mathbf{x})}{1 - P(y=1 \mid \mathbf{x})}
= \mathbf{w}^\top \mathbf{x} + b$$

The probability is recovered with the **sigmoid**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad
P(y=1 \mid \mathbf{x}) = \sigma(\mathbf{w}^\top \mathbf{x} + b)$$

Parameters are learned by minimising the **binary cross-entropy** loss:

$$L(\mathbf{w}, b) = -\frac{1}{n} \sum_{i=1}^{n}
\left[ y_i \log \hat p_i + (1 - y_i) \log(1 - \hat p_i) \right]
+ \alpha \|\mathbf{w}\|^2$$

where $\alpha \geq 0$ controls L2 regularisation strength.

## Dataset Overview

**Source:** UCI Bank Marketing (`fetch_ucirepo(id=222)`) | **Rows used:** 10,000 (random
subsample for tractability) | **Target:** `y` -- did the client subscribe a term deposit?

| Feature group | Examples |
|---|---|
| Demographic | age, job, marital, education |
| Financial | balance, default, housing, loan |
| Contact | contact, day, month, campaign, pdays, previous, poutcome |

The `duration` variable is dropped because its value is only known after the call ends, so
including it would leak the outcome.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

from ucimlrepo import fetch_ucirepo

from mlpackage import (
    LogisticRegression, PCA,
    StandardScaler, train_test_split,
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc,
)

bank   = fetch_ucirepo(id=222)
df     = bank.data.features.copy()
target = bank.data.targets.copy()

# Drop the leaky 'duration' feature (only known after the call)
if "duration" in df.columns:
    df = df.drop(columns=["duration"])

# Subsample 10,000 rows for tractability
combined = df.copy()
combined["y"] = target.values.ravel()
combined = combined.sample(n=10000, random_state=42).reset_index(drop=True)

print(f"Shape: {combined.shape}")
print(f"Columns: {list(combined.columns)}")
combined.head()

## Exploratory Data Analysis

In [ ]:
# Class balance
print("Target distribution:")
print(combined["y"].value_counts())
print(f"Positive rate: {(combined['y'] == 'yes').mean():.4f}")

plt.figure(figsize=(5, 4))
combined["y"].value_counts().plot(kind="bar", color=["steelblue", "coral"])
plt.title("Subscription Outcome Distribution")
plt.ylabel("Count")
plt.xlabel("y")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Numeric feature distributions
numeric_cols = combined.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns: {numeric_cols}")

n_num = len(numeric_cols)
n_cols = 4
n_rows = (n_num + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3 * n_rows))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].hist(combined[col], bins=30, color="steelblue", edgecolor="white")
    axes[i].set_title(col)
for j in range(i + 1, len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation among numeric features
plt.figure(figsize=(8, 6))
sns.heatmap(combined[numeric_cols].corr(), annot=True, cmap="coolwarm",
            center=0, fmt=".2f", cbar_kws={"label": "Correlation"})
plt.title("Numeric Feature Correlation")
plt.tight_layout()
plt.show()

## Preprocessing

- One-hot encode categorical columns with `pd.get_dummies`
- Convert the target string to {0, 1}
- Standardise the numeric matrix (categorical dummies are scaled along with the rest for
  simplicity in this teaching notebook)

In [ ]:
y_bin    = (combined["y"] == "yes").astype(int).values
X_df     = combined.drop(columns=["y"])
X_encoded = pd.get_dummies(X_df, drop_first=True)

print(f"Encoded feature matrix shape: {X_encoded.shape}")
feature_names = X_encoded.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded.values.astype(float), y_bin,
    test_size=0.2, random_state=42,
)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train size: {X_train_s.shape[0]}  |  Test size: {X_test_s.shape[0]}")
print(f"Train positive rate: {y_train.mean():.4f}")

## Model Training

In [ ]:
lr_no_reg = LogisticRegression(alpha=0.0,  learning_rate=0.1, n_iterations=1000)
lr_l2     = LogisticRegression(alpha=1.0,  learning_rate=0.1, n_iterations=1000)

lr_no_reg.fit(X_train_s, y_train)
lr_l2.fit(X_train_s, y_train)

for name, m in [("No regularisation (alpha=0)", lr_no_reg),
                ("L2 regularisation (alpha=1)", lr_l2)]:
    tr = m.score(X_train_s, y_train)
    te = m.score(X_test_s, y_test)
    print(f"{name:35s} Train acc: {tr:.4f}  |  Test acc: {te:.4f}")

## Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, name, m in zip(axes, ["No reg (alpha=0)", "L2 (alpha=1)"], [lr_no_reg, lr_l2]):
    cm = confusion_matrix(y_test, m.predict(X_test_s))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No (0)", "Yes (1)"],
                yticklabels=["No (0)", "Yes (1)"], cbar=False, ax=ax)
    ax.set_title(f"Confusion Matrix -- {name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.tight_layout()
plt.show()

for name, m in [("No regularisation (alpha=0)", lr_no_reg),
                ("L2 regularisation (alpha=1)", lr_l2)]:
    print(f"\n--- {name} ---")
    classification_report(y_test, m.predict(X_test_s))

In [ ]:
# ROC curves
plt.figure(figsize=(7, 6))
for name, m, color in [("No reg",   lr_no_reg, "steelblue"),
                       ("L2",       lr_l2,     "coral")]:
    fpr, tpr, _ = m.roc_curve(X_test_s, y_test)
    auc_val      = m.auc(X_test_s, y_test)
    plt.plot(fpr, tpr, color=color, linewidth=2, label=f"{name} (AUC = {auc_val:.3f})")
plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
plt.title("ROC Curves on Test Set")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Coefficient magnitudes (no-reg model) -- top 15 absolute coefficients
coef = lr_no_reg.coef_
order = np.argsort(np.abs(coef))[::-1][:15]

plt.figure(figsize=(9, 6))
colors = ["steelblue" if coef[i] >= 0 else "coral" for i in order]
plt.barh(np.arange(len(order)), coef[order], color=colors)
plt.yticks(np.arange(len(order)), [feature_names[i] for i in order])
plt.gca().invert_yaxis()
plt.axvline(0, color="black", linewidth=0.5)
plt.title("Top 15 Logistic Regression Coefficients by |magnitude|")
plt.xlabel("Coefficient (positive = increases subscription odds)")
plt.tight_layout()
plt.show()

In [ ]:
# 2-D PCA decision boundary
pca2 = PCA(n_components=2)
pca2.fit(X_train_s)
Z_train = pca2.transform(X_train_s)
Z_test  = pca2.transform(X_test_s)

lr_2d = LogisticRegression(alpha=0.0, learning_rate=0.1, n_iterations=1000)
lr_2d.fit(Z_train, y_train)

xx, yy = np.meshgrid(
    np.linspace(Z_train[:, 0].min() - 1, Z_train[:, 0].max() + 1, 200),
    np.linspace(Z_train[:, 1].min() - 1, Z_train[:, 1].max() + 1, 200),
)
grid    = np.c_[xx.ravel(), yy.ravel()]
zz      = lr_2d.predict_proba(grid)[:, 1].reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, zz, levels=20, cmap="RdBu_r", alpha=0.6)
plt.colorbar(label="P(subscribe)")
plt.scatter(Z_test[y_test == 0, 0], Z_test[y_test == 0, 1], color="blue",
            s=10, alpha=0.4, label="No (0)")
plt.scatter(Z_test[y_test == 1, 0], Z_test[y_test == 1, 1], color="red",
            s=10, alpha=0.6, label="Yes (1)")
plt.title("Logistic Regression Decision Surface (2-D PCA Projection)")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.legend()
plt.tight_layout()
plt.show()

print(f"2D logistic regression test accuracy: {lr_2d.score(Z_test, y_test):.4f}")

## Interpretation and Conclusions

_Analysis to be completed after running the notebook end-to-end._